<a href="https://colab.research.google.com/github/Lingasamy-DS/Eco_cover_type-/blob/main/3_classification_and_EDA_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import numpy as np
import pandas as pd
import pickle
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

warnings.filterwarnings("ignore")

**Reading the datas_encoded(train_data) and the test_data(test_data files) for models training**

In [ ]:
datas_encoded= pd.read_csv("/content/drive/MyDrive/Eco_Forest_Cover_Type_21.06.26/data_encoded.csv")
datas_encoded.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_37,Soil_Type_38,Soil_Type_39,Soil_Type_40,Wilderness_Area_2,Wilderness_Area_3,Wilderness_Area_4,Aspect_Category_North,Aspect_Category_South,Aspect_Category_West
0,0.540904,-1.131123,-0.558045,-0.181155,-0.266110,0.013028,-0.152039,-0.305003,-0.000516,1.823023,...,0,0,0,0,0,0,0,1,0,0
1,-0.266262,0.483868,-0.365606,-0.259596,-0.151381,-0.365541,0.373528,1.343565,0.296428,-0.306397,...,0,0,0,0,0,0,0,0,1,0
2,0.803233,1.242190,-0.183187,2.030259,2.182841,1.585957,-1.279006,1.011305,1.550192,0.336458,...,0,0,0,0,0,0,0,0,0,1
3,0.359292,-0.601573,-1.474177,-0.334966,-0.179827,0.677153,0.131244,0.391227,0.164453,1.454219,...,0,0,0,0,0,0,0,0,0,0
4,1.408607,-1.009703,0.761265,1.080118,2.182841,0.811045,-0.152039,-1.357178,-0.726379,-0.502147,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
test_data = pd.read_csv("/content/drive/MyDrive/Eco_Forest_Cover_Type_21.06.26/test_encoded.csv")
test_data.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_38,Soil_Type_39,Soil_Type_40,Wilderness_Area_2,Wilderness_Area_3,Wilderness_Area_4,Aspect_Category_North,Aspect_Category_South,Aspect_Category_West,Cover_Type
0,0.389560,1.081026,1.166005,1.126385,0.780451,1.360237,-1.917948,1.602846,2.342043,1.308966,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4
1,-0.695069,-0.990634,-0.558045,-0.977229,-0.874924,-0.909994,-0.041272,-0.305003,-0.099497,-1.210251,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4
2,-0.119963,-0.068189,0.761265,-0.627116,0.071429,0.381526,1.652684,-0.434264,-1.353261,0.009640,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4
3,-2.604520,-0.119716,2.108098,0.350553,2.182841,-1.656468,2.330618,-2.181860,-2.508044,-0.714351,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1
4,-0.079605,0.151782,-0.762272,1.054921,0.831072,-0.356062,0.764426,0.694005,-0.132491,-0.179869,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,4


**x and y data splitting**

In [ ]:
X_train= datas_encoded.drop(columns=["Cover_Type"])
y_train= datas_encoded["Cover_Type"]

In [ ]:
X_test = test_data.drop(columns=["Cover_Type"])
y_test = test_data["Cover_Type"]

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(solver='saga',  max_iter = 500),
    "Decision Tree": DecisionTreeClassifier(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost Classifier": XGBClassifier(n_estimators= 200, use_label_encoder=False, eval_metric='mlogloss', objective='multi:softmax')
}

best_model_name = None
best_model_pipeline = None
best_test_accuracy = 0.0

# Loop through models
for name, model in models.items():
    pipeline = Pipeline([
        ('classifier', model)
    ])

    pipeline.fit(X_train, y_train)

    # Make predictions on training data
    y_train_pred = pipeline.predict(X_train)
    # Calculate training accuracy
    train_accuracy = accuracy_score(y_train, y_train_pred)

    # Make predictions on test data
    y_test_pred = pipeline.predict(X_test)
    # Calculate test accuracy
    test_accuracy = accuracy_score(y_test, y_test_pred)

    precision = precision_score(y_test, y_test_pred, average = "weighted", zero_division=0)
    recall= recall_score(y_test, y_test_pred, average = "weighted", zero_division=0)
    f1 = f1_score(y_test, y_test_pred, average = "weighted", zero_division=0)

    # Print results
    print(f"{name}:\n  Training Accuracy = {train_accuracy:.4f}\n  Testing Accuracy = {test_accuracy:.4f}")
    print(f"  Testing Accuracy  = {test_accuracy:.4f}")
    print(f"  Precision         = {precision:.4f}")
    print(f"  Recall            = {recall:.4f}")
    print(f"  F1 Score          = {f1:.4f}\n")

    if test_accuracy > best_test_accuracy:
        best_test_accuracy = test_accuracy
        best_model_name = name
        best_model_pipeline = pipeline

# Save the best model
if best_model_pipeline is not None:
    filename = 'best_model.joblib'
    with open("/content/drive/MyDrive/Eco_Forest_Cover_Type_21.06.26/best_model.pkl", "wb") as f:
      pickle.dump(filename,f)
    print(f"=== Best Model Saved ===\n{best_model_name} with Testing Accuracy: {best_test_accuracy:.4f} saved to '{filename}'")

Logistic Regression:
  Training Accuracy = 0.7706
  Testing Accuracy = 0.6519
  Testing Accuracy  = 0.6519
  Precision         = 0.7840
  Recall            = 0.6519
  F1 Score          = 0.6944

Decision Tree:
  Training Accuracy = 1.0000
  Testing Accuracy = 0.9229
  Testing Accuracy  = 0.9229
  Precision         = 0.9251
  Recall            = 0.9229
  F1 Score          = 0.9237

K-Nearest Neighbors:
  Training Accuracy = 0.9853
  Testing Accuracy = 0.8693
  Testing Accuracy  = 0.8693
  Precision         = 0.8884
  Recall            = 0.8693
  F1 Score          = 0.8747

Random Forest:
  Training Accuracy = 1.0000
  Testing Accuracy = 0.9460
  Testing Accuracy  = 0.9460
  Precision         = 0.9464
  Recall            = 0.9460
  F1 Score          = 0.9461

XGBoost Classifier:
  Training Accuracy = 0.9911
  Testing Accuracy = 0.9374
  Testing Accuracy  = 0.9374
  Precision         = 0.9378
  Recall            = 0.9374
  F1 Score          = 0.9375

=== Best Model Saved ===
Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Hyperparameter grid for Random Forest (from cell '8d71feba')
rf_param_grid = {
    'classifier__n_estimators': [100, 200, 300, 400],
    'classifier__max_features': ['sqrt', 'log2'],
    'classifier__max_depth' : [4,5,6,7,8],
    'classifier__criterion' :['gini', 'entropy'],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4]
}

# Random Forest Pipeline
rf_pipeline = Pipeline([
    ('classifier', RandomForestClassifier(random_state=0))
])

# Perform RandomizedSearchCV for Random Forest on the subset
rf_search_subset = RandomizedSearchCV(rf_pipeline, rf_param_grid, cv=3, verbose=1, n_jobs=-1, n_iter=5, random_state=42)
rf_search_subset.fit(X_train, y_train)

print(f"\nBest parameters for Random Forest (on subset): {rf_search_subset.best_params_}")
print(f"Best cross-validation accuracy for Random Forest (on subset): {rf_search_subset.best_score_:.4f}")

Fitting 3 folds for each of 5 candidates, totalling 15 fits

Best parameters for Random Forest (on subset): {'classifier__n_estimators': 300, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 2, 'classifier__max_features': 'log2', 'classifier__max_depth': 7, 'classifier__criterion': 'gini'}
Best cross-validation accuracy for Random Forest (on subset): 0.8127


In [25]:
!pip install feature_engine

In [26]:
%%writefile "/content/eco.py"
import streamlit as st
import pickle
import pandas as pd
import numpy as np

# Ensure feature_engine is installed when the Streamlit app runs
import subprocess
import sys

try:
    import feature_engine
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "feature_engine"])
    import feature_engine

# Now import from feature_engine
from feature_engine.transformation import YeoJohnsonTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder

# 1. Sets the browser tab title
st.set_page_config(page_title="Eco Forest Type Prediction App")

st.title("Eco Forest Type Prediction")

# -------------------------------
# Load model and transformers
# -------------------------------
try:
    with open("/content/drive/MyDrive/Final Eco/pickle file_1/best_random_forest_model.pkl", "rb") as f:
        model = pickle.load(f)
    with open("/content/drive/MyDrive/Final Eco/pickle file_1/label_encoder (1).pkl", "rb") as f:
        label_encoder = pickle.load(f)
    with open("/content/drive/MyDrive/Final Eco/pickle file_1/yj_transformer.pkl", "rb") as f:
        yj_transformer = pickle.load(f)
    with open("/content/drive/MyDrive/Final Eco/pickle file_1/min_max_scaler.pkl", "rb") as f:
        scaler = pickle.load(f)
    with open("/content/drive/MyDrive/Final Eco/pickle file_1/one_hot_encoder (1).pkl", "rb") as f:
        ohe = pickle.load(f)
except FileNotFoundError as e:
    st.error(f"Error loading a required file: {e}. Please ensure all model and transformer files are saved correctly.")
    st.stop()


# Define feature lists for preprocessing (from the notebook)
yj_cols = ['Aspect', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology',
           'Horizontal_Distance_To_Roadways', 'Horizontal_Distance_To_Fire_Points',
           'Euclidean_Distance_To_Hydrology', 'Hydro_fire_interaction', 'Hillshade_9am', 'Hillshade_Avg']

minmax_cols = ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology',
               'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways',
               'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
               'Horizontal_Distance_To_Fire_Points', 'Euclidean_Distance_To_Hydrology',
               'Hillshade_Avg', 'Hydro_fire_interaction', 'Slop_elevation_interaction']

ohe_cat_cols = ['Soil_Type', 'Wilderness_Area'] # Aspect_Category was not in final_model_cols

# Define the final columns the model expects (from X.columns in the notebook)
final_model_cols = ['Elevation', 'Horizontal_Distance_To_Roadways',
       'Horizontal_Distance_To_Fire_Points', 'Wilderness_Area_4',
       'Hydro_fire_interaction', 'Wilderness_Area_3', 'Soil_Type_10',
       'Horizontal_Distance_To_Hydrology', 'Euclidean_Distance_To_Hydrology',
       'Soil_Type_3', 'Hillshade_Avg', 'Hillshade_9am', 'Aspect',
       'Hillshade_3pm', 'Hillshade_Noon', 'Vertical_Distance_To_Hydrology',
       'Slop_elevation_interaction', 'Soil_Type_38', 'Slope', 'Soil_Type_29']


# -------------------------------
# Define input fields (raw features)
# -------------------------------
st.header("Enter Feature Values:")

# Numerical Inputs
Elevation = st.number_input("Elevation (meters)", min_value=0.0, max_value=float(7000), value=2596.0)
Aspect = st.number_input("Aspect (degrees)", min_value=0.0, max_value=360.0, value=51.0)
Slope = st.number_input("Slope (degrees)", min_value=0.0, max_value=float(100), value=3.0)
Horizontal_Distance_To_Hydrology = st.number_input("Horizontal Distance To Hydrology (meters)", min_value=0.0, max_value=float(5000), value=258.0)
Vertical_Distance_To_Hydrology = st.number_input("Vertical Distance To Hydrology (meters)", min_value=float(-1000), max_value=float(1000), value=0.0)
Horizontal_Distance_To_Roadways = st.number_input("Horizontal Distance To Roadways (meters)", min_value=0.0, max_value=float(100000), value=510.0)
Hillshade_9am = st.number_input("Hillshade 9am (0-255)", min_value=0.0, max_value=255.0, value=221.0)
Hillshade_Noon = st.number_input("Hillshade Noon (0-255)", min_value=0.0, max_value=255.0, value=232.0)
Hillshade_3pm = st.number_input("Hillshade 3pm (0-255)", min_value=0.0, max_value=255.0, value=148.0)
Horizontal_Distance_To_Fire_Points = st.number_input("Horizontal Distance To Fire Points (meters)", min_value=0.0, max_value=float(10000), value=6279.0)
Euclidean_Distance_To_Hydrology = st.number_input("Euclidean Distance To Hydrology (meters)", min_value=0.0, max_value=float(2000), value=258.0)
Hillshade_Avg = st.number_input("Hillshade Average (0-255)", min_value=0.0, max_value=255.0, value=200.0)
Hydro_fire_interaction = st.number_input("Hydro Fire Interaction (meters)", min_value=0.0, max_value=float(10000), value=6021.0)
Slop_elevation_interaction = st.number_input("Slope Elevation Interaction (meters)", min_value=0.0, max_value=float(300000), value=7788.0)

# Categorical Inputs (using common options or ranges)
# Wilderness_Area values from original dataset: 1, 2, 3, 4
Wilderness_Area = st.selectbox("Wilderness Area", [1, 2, 3, 4], index=0)

# Soil_Type values from original dataset: 1-40
Soil_Type = st.selectbox("Soil Type", list(range(1, 41)), index=28) # Default to 29, as seen in df_resampled.head()

# -------------------------------
# Prediction
# -------------------------------
if st.button("Predict Cover Type"):
    # Create a DataFrame from raw inputs
    raw_input_df = pd.DataFrame({
        'Elevation': [Elevation],
        'Aspect': [Aspect],
        'Slope': [Slope],
        'Horizontal_Distance_To_Hydrology': [Horizontal_Distance_To_Hydrology],
        'Vertical_Distance_To_Hydrology': [Vertical_Distance_To_Hydrology],
        'Horizontal_Distance_To_Roadways': [Horizontal_Distance_To_Roadways],
        'Hillshade_9am': [Hillshade_9am],
        'Hillshade_Noon': [Hillshade_Noon],
        'Hillshade_3pm': [Hillshade_3pm],
        'Horizontal_Distance_To_Fire_Points': [Horizontal_Distance_To_Fire_Points],
        'Euclidean_Distance_To_Hydrology': [Euclidean_Distance_To_Hydrology],
        'Hillshade_Avg': [Hillshade_Avg],
        'Hydro_fire_interaction': [Hydro_fire_interaction],
        'Slop_elevation_interaction': [Slop_elevation_interaction],
        'Wilderness_Area': [Wilderness_Area],
        'Soil_Type': [Soil_Type]
    })

    # Apply Yeo-Johnson Transformation
    input_yj_transformed = raw_input_df.copy()
    if not input_yj_transformed[yj_cols].empty:
        input_yj_transformed[yj_cols] = yj_transformer.transform(input_yj_transformed[yj_cols])

    # Apply Min-Max Scaling
    input_scaled = input_yj_transformed.copy()
    if not input_scaled[minmax_cols].empty:
        input_scaled[minmax_cols] = scaler.transform(input_scaled[minmax_cols])

    # Apply One-Hot Encoding
    input_ohe_part = ohe.transform(raw_input_df[ohe_cat_cols])
    ohe_feature_names = ohe.get_feature_names_out(ohe_cat_cols)
    input_ohe_df = pd.DataFrame(input_ohe_part, columns=ohe_feature_names, index=raw_input_df.index)

    # Combine numerical (scaled) and OHE features
    # Drop original categorical columns from input_scaled before concatenating OHE features
    numerical_features_for_concat = input_scaled.drop(columns=ohe_cat_cols)
    processed_input_df = pd.concat([numerical_features_for_concat, input_ohe_df], axis=1)

    # Select and reorder columns to match the model's expected input
    # Handle missing columns by adding them with 0, and extra columns by dropping them
    final_input_df = pd.DataFrame(columns=final_model_cols)
    for col in final_model_cols:
        if col in processed_input_df.columns:
            final_input_df[col] = processed_input_df[col]
        else:
            final_input_df[col] = 0.0 # Add missing columns with default value 0

    # Ensure the order is correct
    final_input_df = final_input_df[final_model_cols]

    prediction_numerical = model.predict(final_input_df)[0]
    predicted_cover_type = label_encoder.inverse_transform([int(prediction_numerical)])[0]
    st.success(f"Predicted Cover Type: {predicted_cover_type}")

Overwriting /content/eco.py


In [29]:
!streamlit run /content/eco.py &>/content/logs.txt &

###### **Executing the following code will generate a URL. Click it or copy and paste it in browser to view your streamlit page**

In [28]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!pkill -f cloudflared
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 > nohup.out 2>&1 &

--2026-06-23 11:47:47--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.6.1/cloudflared-linux-amd64 [following]
--2026-06-23 11:47:48--  https://github.com/cloudflare/cloudflared/releases/download/2026.6.1/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/74ac6867-a1bf-4352-b955-c16fbb86d0f6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-23T12%3A42%3A22Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-23T1

In [30]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your new tunnel URL is: {}"

Your new tunnel URL is: https://bingo-entitled-testimony-disco.trycloudflare.com


In [27]:
!pip install streamlit streamlit_option_menu  # installing streamlit and streamlit_option_menu packages

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://bizarre-rogers-impact-technique.trycloudflare.com
